# Criação de arquivos

Criando os 5 arquivos vazios correspondente a cada uma das 5 etapas. 

In [1]:
for i in range(1, 6):
    with open(f"etapa-{i}.txt", "w", encoding="utf-8") as f:
        pass

# Leitura dos dados
Abrindo o csv para a leitura de forma segura

In [2]:
with open("actors.csv", "r", encoding="utf-8") as f:
    csv = f.readlines()

# Tratamento dos dados

Para armazenar o csv, optei por utilizar um dicionário. 

Houve um problema: alguns dados de string estão entre '"' e possuem o cactere ','.
Ao fazer ".strip()", a string seria dividida incorretamente, fazendo com que os próximos campos ficassem com os dados nas posições erradas. 

Para isso criei um _parser_ que recebe uma linha, e, ao encontrar virgula, verifica se houve abertura de aspas antes de dividir. 
Ao encontrar ',' a string só será dividida, se não estiver dentro de aspas.

Dessa forma, os dados da linha serão fatiadas corretamente e retornados.

In [3]:
def formatar_linha(linha: list) -> list:
    
    linha_formatada = []
    dado_atual = ''
    dentro_aspas = False

    for caractere in linha:

        if caractere == '"':

            dentro_aspas = not dentro_aspas   #inverte o estado atual da flag dentro_aspas

        elif caractere == ',' and not dentro_aspas:

            linha_formatada.append(dado_atual.strip())  #faz o strip se achar virgula e se a flag das aspas estiver em false. Limpa o dado_atual
            dado_atual = ''

        else:
            dado_atual += caractere   # concatena o proximo caractere
            
    linha_formatada.append(dado_atual.strip()) 

    return linha_formatada

# Criação da estrutura de dados

Usando a função de formatar a linha, extraimos o cabeçalho referente as colunas do csv, que serão usados como chaves no dicionário

In [4]:
cabecalho = formatar_linha(csv[0])

Em seguida, é criado uma lista de linhas para armazenar os dados de cada linha do csv, extraindo seus campos corretamente, utilizando a função "formatar_linha".

In [5]:
dados = [formatar_linha(linha) for linha in csv[1:]]

É criado o dicionário como a estrutura de dados, onde cada chave será uma coluna do cabeçalho e seu conteúdo será o campo correspondente para aquela coluna.

In [6]:
dc = [dict(zip(cabecalho, linha)) for linha in dados]

# Etapa 1

### Apresente o ator/atriz com maior número de filmes e a respectiva quantidade. A quantidade de filmes encontra-se na coluna 'Number of movies' do arquivo.

Para isso, foram criadas as variáveis temporárias para armazenar o nome e a quantidade de filmes do autor com maior número de filmes.
É feito um laço percorrendo o dicionário com a chave 'Number of Movies' e é feita sua conversão para int (ignorando se falhar), e verificando
se a quantidade atual é maior do que a armezanada. Se for, altera as variaveis temporárias e no final, escreve o maior encontrado no arquivo

In [7]:
nome_ator_mais_filmes = ""
maior_qtd_filmes = -1

for linha in dc:

    try:

        qtd = int(linha['Number of Movies'])

        if qtd > maior_qtd_filmes:

            maior_qtd_filmes = qtd
            nome_ator_mais_filmes = linha['Actor']

    except ValueError:
        continue

with open("etapa-1.txt", "w", encoding="utf-8") as f:
    f.write(f"O ator que mais possui filmes no dataset é {nome_ator_mais_filmes} com {maior_qtd_filmes} filmes.")


# Etapa 2

### Apresente a média de receita de bilheteria bruta dos principais filmes, considerando todos os atores. Estamos falando aqui da média da coluna Gross.

É criada uma lista vazia que vai armazenar cada receita, e também como anteriormente, é feito um laço percorrendo o dicionário com a chave 'Gross' armazenando cada uma numa lista. 
Ao final, é feita a soma de todos os elementos da lista e divide-se pela quantidade de elementos para fazer a média. O valor obtido é escrito no arquivo correspondente a etapa 2.

In [8]:
lista_receitas = []

for linha in dc:

    try:
        valor = float(linha['Gross'])
        lista_receitas.append(valor)

    except ValueError:
        continue

receita_media = sum(lista_receitas) / len(lista_receitas)

with open("etapa-2.txt", "w", encoding="utf-8") as f:
    f.write(f"A média de receita bruta dos principais filmes é ${receita_media:.2f} milhões.")

# Etapa 3

### Apresente o ator/atriz com a maior média de receita de bilheteria bruta por filme do conjunto de dados. Considere a coluna 'Average per Movie' para fins de cálculo.

Exatamente a mesma lógica da etapa 1, usando a chave 'Average per Movie'.

In [9]:
nome_ator_maior_receita_media = ""
max_media = -1

for linha in dc:

    try:

        valor = float(linha['Average per Movie'])

        if valor > max_media:   
            nome_ator_maior_receita_media = linha['Actor']
            max_media = valor

    except ValueError:
        continue

with open("etapa-3.txt", "w", encoding="utf-8") as f:
    f.write(f"O ator com a maior média de bilheteria por filme é {nome_ator_maior_receita_media}, com média de ${max_media:.2f} milhões por filme.")

# Etapa 4

### A coluna '#1 Movie' contém o filme de maior bilheteria em que o ator atuou. Realize a contagem de aparições destes filmes no dataset, listando-os ordenados pela quantidade de vezes em que estão presentes. Considere a ordem decrescente e, em segundo nível, o nome do filme. 

### Ao escrever no arquivo, considere o padrão de saída:

### "(sequencia) - O filme (nome filme) aparece (quantidade)vez(es) no dataset", 

### adicionando um resultado a cada linha.


Nesse requisito, foi pedido a "sequencia" do filme no qual não é uma das chaves do nosso diocionário. Para obtê-la, será necessário extrair a partir do nome.
Nesses dados, aqueles com ':' e '/', mencionam a sequência que pertencem. Os filmes que possuem um número ao seu fim, são uma continuação (logo, possuem uma sequencia de filmes).

Obs.: Optei por deixar o título do filme como sendo o nome completo.

Com essas premissas, no primeiro caso, basta verificar se há algum dos simbolos no nome do filme. Se houver, faz o split onde encontrar o separador. Em seguida, extrai-se 
o nome da sequencia da primeira parte dividida. 

No caso de não haver nenhum dos simbolos, verificar se o ultimo caractere da string é um dígito. Se for, a sequencia será o nome do filme sem o dígito.

In [10]:
def extrair_sequencia_e_titulo(nome_filme: str) -> tuple:
    nome_filme = nome_filme.strip()

    for sep in [":", "/"]:

        if sep in nome_filme:

            partes = nome_filme.split(sep, 1)
            sequencia = partes[0].strip()

            return sequencia, nome_filme

    partes = nome_filme.split()

    if partes and partes[-1].isdigit():

        sequencia = " ".join(partes[:-1])

        return sequencia, nome_filme

    return nome_filme, nome_filme

É criado então, um dicionário temporário 'filmes' onde suas chaves serão os nomes de cada filme, isso porque um filme vai aparecer mais de uma vez. 
Para a contagem, o nome extraído como chave é usado a função "get()" para retornar o valor atual da chave, ou '0' se não houver. Ao final, é somado +1 para a contagem.
Após percorrer todos os dados, é feito a ordenação decrescente (-x[1]) do dicionário temporário, a partir dos valores contados.
Após a abertura do arquivo, para cada item da lista ordenada, é usado a função de extrair a sequencia e título a partir do seu nome. 
São escritos linha-a-linha, a sequencia, titulo e quantidade que apareceram no arquivo.


In [11]:
filmes = {}

for linha in dc:

    nome_filme = linha['#1 Movie']
    filmes[nome_filme] = filmes.get(nome_filme, 0) + 1   #para fazer a contagem 

ordenado = sorted(filmes.items(), key=lambda x: (-x[1], x[0]))

with open("etapa-4.txt", "w", encoding="utf-8") as f:
    
    for nome, qtd in ordenado:
        
        sequencia, titulo = extrair_sequencia_e_titulo(nome)

        f.write(f"{sequencia} - O filme \"{titulo}\" aparece {qtd} vez(es) no dataset.\n")

# Etapa 5

### Apresente a lista dos atores ordenada pela receita bruta de bilheteria de seus filmes (coluna Total Gross), em ordem decrescente. Ao escrever no arquivo, considere o padrão de saída (nome do ator) - (receita total bruta), adicionando um resultado a cada linha.

Como a receita dos atores estavam em milhões, e todos possuiam mais de 1000, decidi fazer a formatação do valor para bilhões, apenas para fins de boa legibilidade.
Recebe o valor de float referente aos milhões, e retorna a string convertida e formatada refente ao valor.

In [12]:
def formatar_valor_milhoes_para_bilhoes(valor_em_milhoes: float) -> str:
    if valor_em_milhoes >= 1000:
        valor_em_bilhoes = valor_em_milhoes / 1000
        return f"${valor_em_bilhoes:.3f} bilhões"
    else:
        return f"${valor_em_milhoes:.2f} milhões"

Como o dicionario de dados não possuem atores repetidos, vamos usar uma lista temporária para ir armazanando cada nome de ator e sua renda bruta (fazendo a conversão de String para float). 
Ao final, essa lista temporária é ordenada, e por cada linha da lista, é escrita o nome, e a renda bruta total dos atores, usando a função de formatação de milhoes para bilhoes.

In [13]:
atores = []

for linha in dc:

    try:

        nome = linha["Actor"]
        total = float(linha["Total Gross"])
        atores.append((nome, total))

    except ValueError:
        continue

ordenado = sorted(atores, key=lambda x: -x[1])

with open("etapa-5.txt", "w", encoding="utf-8") as f:
    
    for nome, total in ordenado:
        f.write(f"{nome} - {formatar_valor_milhoes_para_bilhoes(total)}\n")